In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv(r"C:\Users\arnaw\OneDrive\Desktop\transfer_recomender\data\players_data-2024_2025.csv")
print("Shape:", df.shape)
df.head()

Shape: (2854, 267)


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,Att (GK),Thr,Launch%,AvgLen,Opp,Stp,Stp%,#OPA,#OPA/90,AvgDist
0,1,Max Aarons,eng ENG,DF,Bournemouth,eng Premier League,24.0,2000.0,3,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Max Aarons,eng ENG,"DF,MF",Valencia,es La Liga,24.0,2000.0,4,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Rodrigo Abajas,es ESP,DF,Valencia,es La Liga,21.0,2003.0,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,James Abankwah,ie IRL,"DF,MF",Udinese,it Serie A,20.0,2004.0,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Keyliane Abdallah,fr FRA,FW,Marseille,fr Ligue 1,18.0,2006.0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df['Primary_Pos'] = df['Pos'].str.split(',').str[0]

print(df['Primary_Pos'].unique())
print(df['Primary_Pos'].value_counts())

['DF' 'FW' 'MF' 'GK']
Primary_Pos
DF    1022
MF     900
FW     720
GK     212
Name: count, dtype: int64


In [4]:
gk_cols = [col for col in df.columns if 'keeper' in col.lower()]
gk_cols += ['CS%', 'PSxG/SoT', 'Save%', 'Saves', 'GA', 'GA90', 
             'SoTA', 'W', 'D', 'L', 'PKA', 'PKsv', 'PKm', 
             'AvgDist', 'Stp%', 'Stp', 'Thr', 'Launch%', 'AvgLen',
             'Opp', 'AttGK']


gk_cols = list(set(gk_cols))

df.drop(columns=[col for col in gk_cols if col in df.columns], inplace=True)

print("Remaining columns:", df.shape[1])

Remaining columns: 222


In [5]:
gk_df = df[df['Primary_Pos'] == 'GK'].copy()
outfield_df = df[df['Primary_Pos'] != 'GK'].copy()

print("Goalkeepers:", len(gk_df))
print("Outfield players:", len(outfield_df))

Goalkeepers: 212
Outfield players: 2642


In [21]:
print("NaNs before filling:")
print(outfield_df.isnull().sum().sort_values(ascending=False).head(10))

outfield_df.fillna(0, inplace=True)

print("\nNaNs after filling:")
print(outfield_df.isnull().sum().sum())

NaNs before filling:
#OPA        2642
/90         2642
CS          2642
PSxG+/-     2642
Att (GK)    2642
#OPA/90     2642
PSxG        2642
G/SoT        676
Mn/Start     355
Dist         315
dtype: int64

NaNs after filling:
0


In [9]:
fw_features = [
    'Goals', 'Ast', 'Sh', 'SoT', 'xG', 'npxG', 'G/Sh',
    'KP', 'PPA', 'CrsPA', 'PrgC', 'PrgP', 'Touches'
]

mf_features = [
    'Ast', 'KP', 'PPA', 'PrgP', 'PrgC', 'Touches',
    'TklW', 'Int', 'Blocks', 'Clr', 'xA', 'SCA', 'GCA'
]

df_features = [
    'TklW', 'Int', 'Blocks', 'Clr', 'Err',
    'Won', 'Lost', 'PrgP', 'PrgC', 'Touches', 'AerWon%'
]


fw_features = [col for col in fw_features if col in outfield_df.columns]
mf_features = [col for col in mf_features if col in outfield_df.columns]
df_features = [col for col in df_features if col in outfield_df.columns]

print("FW features:", fw_features)
print("MF features:", mf_features)
print("DF features:", df_features)

FW features: ['Ast', 'Sh', 'SoT', 'xG', 'npxG', 'G/Sh', 'KP', 'PPA', 'CrsPA', 'PrgC', 'PrgP', 'Touches']
MF features: ['Ast', 'KP', 'PPA', 'PrgP', 'PrgC', 'Touches', 'TklW', 'Int', 'Blocks', 'Clr', 'xA', 'SCA', 'GCA']
DF features: ['TklW', 'Int', 'Blocks', 'Clr', 'Err', 'Won', 'Lost', 'PrgP', 'PrgC', 'Touches']


In [10]:
scaler = StandardScaler()

fw_df = outfield_df[outfield_df['Primary_Pos'] == 'FW'].copy()
mf_df = outfield_df[outfield_df['Primary_Pos'] == 'MF'].copy()
df_df = outfield_df[outfield_df['Primary_Pos'] == 'DF'].copy()

fw_df[fw_features] = scaler.fit_transform(fw_df[fw_features])
mf_df[mf_features] = scaler.fit_transform(mf_df[mf_features])
df_df[df_features] = scaler.fit_transform(df_df[df_features])

print("FW players:", len(fw_df))
print("MF players:", len(mf_df))
print("DF players:", len(df_df))

FW players: 720
MF players: 900
DF players: 1022


In [7]:
fw_df.to_csv(r"C:\Users\arnaw\OneDrive\Desktop\transfer_recomender\data\fw_cleaned.csv", index=False)
mf_df.to_csv(r"C:\Users\arnaw\OneDrive\Desktop\transfer_recomender\data\mf_cleaned.csv", index=False)
df_df.to_csv(r"C:\Users\arnaw\OneDrive\Desktop\transfer_recomender\data\df_cleaned.csv", index=False)


outfield_df.to_csv(r"C:\Users\arnaw\OneDrive\Desktop\transfer_recomender\data\outfield_cleaned.csv", index=False)

print("All files saved successfully! ✅")

All files saved successfully! ✅
